I first explored the Northwind source database and understood the main tables and their relationships.
i decided to build a star schema with one fact table and multiple dimension tables.
The grain of our fact table will be one row per order line (one product within an order).
i exported the five source tables as CSV files and set up Databricks for the project.
Next, ill load these raw files into the Bronze layer using PySpark.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.northwind;

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.northwind.raw_files
""")

DataFrame[]

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
company
company1
default
information_schema
northwind
sales
salesdb
salesdwh


### BRONZE LAYER

creating a bronze layer so that we can store the original copy of our data

In [0]:
customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/northwind/raw_files/customer.csv")
)

display(customer_df)

custId,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,mobile,email,fax
1,Customer NRZBB,"Allen, Michael",Sales Representative,Obere Str. 0123,Berlin,NULL,10092,Germany,030-3456789,NULL,NULL,030-0123456
2,Customer MLTDN,"Hassall, Mark",Owner,Avda. de la Constitución 5678,México D.F.,NULL,10077,Mexico,(5) 789-0123,NULL,NULL,(5) 456-7890
3,Customer KBUDE,"Peoples, John",Owner,Mataderos 7890,México D.F.,NULL,10097,Mexico,(5) 123-4567,NULL,NULL,NULL
4,Customer HFBZG,"Arndt, Torsten",Sales Representative,7890 Hanover Sq.,London,NULL,10046,UK,(171) 456-7890,NULL,NULL,(171) 456-7891
5,Customer HGVLZ,"Higginbotham, Tom",Order Administrator,Berguvsvägen 5678,Luleå,NULL,10112,Sweden,0921-67 89 01,NULL,NULL,0921-23 45 67
6,Customer XHXJV,"Poland, Carole",Sales Representative,Forsterstr. 7890,Mannheim,NULL,10117,Germany,0621-67890,NULL,NULL,0621-12345
7,Customer QXVLA,"Bansal, Dushyant",Marketing Manager,"2345, place Kléber",Strasbourg,NULL,10089,France,67.89.01.23,NULL,NULL,67.89.01.24
8,Customer QUHWH,"Ilyina, Julia",Owner,"C/ Araquil, 0123",Madrid,NULL,10104,Spain,(91) 345 67 89,NULL,NULL,(91) 012 34 56
9,Customer RTXGC,"Raghav, Amritansh",Owner,"6789, rue des Bouchers",Marseille,NULL,10105,France,23.45.67.89,NULL,NULL,23.45.67.80
10,Customer EEALV,"Bassols, Pilar Colome",Accounting Manager,8901 Tsawassen Blvd.,Tsawassen,BC,10111,Canada,(604) 901-2345,NULL,NULL,(604) 678-9012


In [0]:
#checking teh schema 
customer_df.printSchema()

root
 |-- custId: integer (nullable = true)
 |-- companyName: string (nullable = true)
 |-- contactName: string (nullable = true)
 |-- contactTitle: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postalCode: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- mobile: string (nullable = true)
 |-- email: string (nullable = true)
 |-- fax: string (nullable = true)



#### schema looks fine and matches the source table

In [0]:
#saving it in the bronze table
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.bronze_customer")

In [0]:
df_salesorder = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/northwind/raw_files/salesorder.csv")
)

display(df_salesorder)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
10248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,NULL,10345,France
10249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,NULL,10328,Germany
10250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
10251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,NULL,10342,France
10252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,NULL,10318,Belgium
10253,34,3,2006-07-10T00:00:00.000Z,2006-07-24T00:00:00.000Z,2006-07-16 00:00:00,2,58.17,Destination JPAIY,"Rua do Paço, 8901",Rio de Janeiro,RJ,10196,Brazil
10254,14,5,2006-07-11T00:00:00.000Z,2006-08-08T00:00:00.000Z,2006-07-23 00:00:00,2,22.98,Destination YUJRD,Hauptstr. 1234,Bern,NULL,10139,Switzerland
10255,68,9,2006-07-12T00:00:00.000Z,2006-08-09T00:00:00.000Z,2006-07-15 00:00:00,3,148.33,Ship to 68-A,Starenweg 6789,Genève,NULL,10294,Switzerland
10256,88,3,2006-07-15T00:00:00.000Z,2006-08-12T00:00:00.000Z,2006-07-17 00:00:00,2,13.97,Ship to 88-B,"Rua do Mercado, 5678",Resende,SP,10354,Brazil
10257,35,4,2006-07-16T00:00:00.000Z,2006-08-13T00:00:00.000Z,2006-07-22 00:00:00,3,81.91,Destination JYDLM,Carrera1234 con Ave. Carlos Soublette #8-35,San Cristóbal,Táchira,10199,Venezuela


In [0]:
df_salesorder.printSchema()

root
 |-- orderId: integer (nullable = true)
 |-- custId: integer (nullable = true)
 |-- employeeId: integer (nullable = true)
 |-- orderDate: timestamp (nullable = true)
 |-- requiredDate: timestamp (nullable = true)
 |-- shippedDate: string (nullable = true)
 |-- shipperid: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- shipName: string (nullable = true)
 |-- shipAddress: string (nullable = true)
 |-- shipCity: string (nullable = true)
 |-- shipRegion: string (nullable = true)
 |-- shipPostalCode: integer (nullable = true)
 |-- shipCountry: string (nullable = true)



In [0]:
df_salesorder.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.bronze_salesorder")

In [0]:
orderdetail_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/northwind/raw_files/orderdetail.csv")
)

display(orderdetail_df)

orderDetailId,orderId,productId,unitPrice,quantity,discount
1,10248,11,14.0,12,0.0
2,10248,42,9.8,10,0.0
3,10248,72,34.8,5,0.0
4,10249,14,18.6,9,0.0
5,10249,51,42.4,40,0.0
6,10250,41,7.7,10,0.0
7,10250,51,42.4,35,0.15
8,10250,65,16.8,15,0.15
9,10251,22,16.8,6,0.05
10,10251,57,15.6,15,0.05


In [0]:
orderdetail_df.printSchema()

root
 |-- orderDetailId: integer (nullable = true)
 |-- orderId: integer (nullable = true)
 |-- productId: integer (nullable = true)
 |-- unitPrice: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



In [0]:
#creating the bronze table
orderdetail_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.bronze_orderdetail")

In [0]:
product_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/northwind/raw_files/product.csv")
)

display(product_df)

productId,productName,supplierId,categoryId,quantityPerUnit,unitPrice,unitsInStock,unitsOnOrder,reorderLevel,discontinued
1,Product HHYDP,1,1,NULL,18.0,NULL,NULL,NULL,0
2,Product RECZE,1,1,NULL,19.0,NULL,NULL,NULL,0
3,Product IMEHJ,1,2,NULL,10.0,NULL,NULL,NULL,0
4,Product KSBRM,2,2,NULL,22.0,NULL,NULL,NULL,0
5,Product EPEIM,2,2,NULL,21.35,NULL,NULL,NULL,1
6,Product VAIIV,3,2,NULL,25.0,NULL,NULL,NULL,0
7,Product HMLNI,3,7,NULL,30.0,NULL,NULL,NULL,0
8,Product WVJFP,3,2,NULL,40.0,NULL,NULL,NULL,0
9,Product AOZBW,4,6,NULL,97.0,NULL,NULL,NULL,1
10,Product YHXGE,4,8,NULL,31.0,NULL,NULL,NULL,0


In [0]:
product_df.printSchema()

root
 |-- productId: integer (nullable = true)
 |-- productName: string (nullable = true)
 |-- supplierId: integer (nullable = true)
 |-- categoryId: integer (nullable = true)
 |-- quantityPerUnit: string (nullable = true)
 |-- unitPrice: double (nullable = true)
 |-- unitsInStock: string (nullable = true)
 |-- unitsOnOrder: string (nullable = true)
 |-- reorderLevel: string (nullable = true)
 |-- discontinued: integer (nullable = true)



In [0]:
product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.bronze_product")

In [0]:
category_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/northwind/raw_files/category.csv")
)

display(category_df)

categoryId,categoryName,description,picture
1,Beverages,"Soft drinks, coffees, teas, beers, and ales",NULL
2,Condiments,"Sweet and savory sauces, relishes, spreads, and seasonings",NULL
3,Confections,"Desserts, candies, and sweet breads",NULL
4,Dairy Products,Cheeses,NULL
5,Grains/Cereals,"Breads, crackers, pasta, and cereal",NULL
6,Meat/Poultry,Prepared meats,NULL
7,Produce,Dried fruit and bean curd,NULL
8,Seafood,Seaweed and fish,NULL


In [0]:
category_df.printSchema()

root
 |-- categoryId: integer (nullable = true)
 |-- categoryName: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)



In [0]:
category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.bronze_category")

### Silver table :-

 Bronze table was created succesfully ,now we will create the silver table where we will focus on data cleaning and analysis

In [0]:
customer_bronze = spark.table("workspace.northwind.bronze_customer")
display(customer_bronze)

custId,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,mobile,email,fax
1,Customer NRZBB,"Allen, Michael",Sales Representative,Obere Str. 0123,Berlin,NULL,10092,Germany,030-3456789,NULL,NULL,030-0123456
2,Customer MLTDN,"Hassall, Mark",Owner,Avda. de la Constitución 5678,México D.F.,NULL,10077,Mexico,(5) 789-0123,NULL,NULL,(5) 456-7890
3,Customer KBUDE,"Peoples, John",Owner,Mataderos 7890,México D.F.,NULL,10097,Mexico,(5) 123-4567,NULL,NULL,NULL
4,Customer HFBZG,"Arndt, Torsten",Sales Representative,7890 Hanover Sq.,London,NULL,10046,UK,(171) 456-7890,NULL,NULL,(171) 456-7891
5,Customer HGVLZ,"Higginbotham, Tom",Order Administrator,Berguvsvägen 5678,Luleå,NULL,10112,Sweden,0921-67 89 01,NULL,NULL,0921-23 45 67
6,Customer XHXJV,"Poland, Carole",Sales Representative,Forsterstr. 7890,Mannheim,NULL,10117,Germany,0621-67890,NULL,NULL,0621-12345
7,Customer QXVLA,"Bansal, Dushyant",Marketing Manager,"2345, place Kléber",Strasbourg,NULL,10089,France,67.89.01.23,NULL,NULL,67.89.01.24
8,Customer QUHWH,"Ilyina, Julia",Owner,"C/ Araquil, 0123",Madrid,NULL,10104,Spain,(91) 345 67 89,NULL,NULL,(91) 012 34 56
9,Customer RTXGC,"Raghav, Amritansh",Owner,"6789, rue des Bouchers",Marseille,NULL,10105,France,23.45.67.89,NULL,NULL,23.45.67.80
10,Customer EEALV,"Bassols, Pilar Colome",Accounting Manager,8901 Tsawassen Blvd.,Tsawassen,BC,10111,Canada,(604) 901-2345,NULL,NULL,(604) 678-9012


In [0]:
#checking for null values
from pyspark.sql.functions import col, count, when

# Check null values in each column
null_check = customer_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customer_bronze.columns
])

display(null_check)

custId,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,mobile,email,fax
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
#chceking duplicates
duplicate_customers = (
    customer_bronze
    .groupBy("custId")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_customers)

custId,count


### Insight :-

there were no null / duplicate values so we can move forward

In [0]:
#standardizing the data and creating a silver layer
customer_silver = customer_bronze.select(
    col("custId").alias("customer_id"),
    col("companyName").alias("company_name"),
    col("contactName").alias("contact_name"),
    col("contactTitle").alias("contact_title"),
    col("address"),
    col("city"),
    col("region"),
    col("postalCode").alias("postal_code"),
    col("country"),
    col("phone"),
    col("mobile"),
    col("email"),
    col("fax")
)

display(customer_silver)

customer_id,company_name,contact_name,contact_title,address,city,region,postal_code,country,phone,mobile,email,fax
1,Customer NRZBB,"Allen, Michael",Sales Representative,Obere Str. 0123,Berlin,NULL,10092,Germany,030-3456789,NULL,NULL,030-0123456
2,Customer MLTDN,"Hassall, Mark",Owner,Avda. de la Constitución 5678,México D.F.,NULL,10077,Mexico,(5) 789-0123,NULL,NULL,(5) 456-7890
3,Customer KBUDE,"Peoples, John",Owner,Mataderos 7890,México D.F.,NULL,10097,Mexico,(5) 123-4567,NULL,NULL,NULL
4,Customer HFBZG,"Arndt, Torsten",Sales Representative,7890 Hanover Sq.,London,NULL,10046,UK,(171) 456-7890,NULL,NULL,(171) 456-7891
5,Customer HGVLZ,"Higginbotham, Tom",Order Administrator,Berguvsvägen 5678,Luleå,NULL,10112,Sweden,0921-67 89 01,NULL,NULL,0921-23 45 67
6,Customer XHXJV,"Poland, Carole",Sales Representative,Forsterstr. 7890,Mannheim,NULL,10117,Germany,0621-67890,NULL,NULL,0621-12345
7,Customer QXVLA,"Bansal, Dushyant",Marketing Manager,"2345, place Kléber",Strasbourg,NULL,10089,France,67.89.01.23,NULL,NULL,67.89.01.24
8,Customer QUHWH,"Ilyina, Julia",Owner,"C/ Araquil, 0123",Madrid,NULL,10104,Spain,(91) 345 67 89,NULL,NULL,(91) 012 34 56
9,Customer RTXGC,"Raghav, Amritansh",Owner,"6789, rue des Bouchers",Marseille,NULL,10105,France,23.45.67.89,NULL,NULL,23.45.67.80
10,Customer EEALV,"Bassols, Pilar Colome",Accounting Manager,8901 Tsawassen Blvd.,Tsawassen,BC,10111,Canada,(604) 901-2345,NULL,NULL,(604) 678-9012


In [0]:
#saving it into the silver table 
customer_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.silver_customer")

### Silver table :- salesorder

In [0]:
salesorder_bronze = spark.table("workspace.northwind.bronze_salesorder")

display(salesorder_bronze)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
10248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,NULL,10345,France
10249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,NULL,10328,Germany
10250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
10251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,NULL,10342,France
10252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,NULL,10318,Belgium
10253,34,3,2006-07-10T00:00:00.000Z,2006-07-24T00:00:00.000Z,2006-07-16 00:00:00,2,58.17,Destination JPAIY,"Rua do Paço, 8901",Rio de Janeiro,RJ,10196,Brazil
10254,14,5,2006-07-11T00:00:00.000Z,2006-08-08T00:00:00.000Z,2006-07-23 00:00:00,2,22.98,Destination YUJRD,Hauptstr. 1234,Bern,NULL,10139,Switzerland
10255,68,9,2006-07-12T00:00:00.000Z,2006-08-09T00:00:00.000Z,2006-07-15 00:00:00,3,148.33,Ship to 68-A,Starenweg 6789,Genève,NULL,10294,Switzerland
10256,88,3,2006-07-15T00:00:00.000Z,2006-08-12T00:00:00.000Z,2006-07-17 00:00:00,2,13.97,Ship to 88-B,"Rua do Mercado, 5678",Resende,SP,10354,Brazil
10257,35,4,2006-07-16T00:00:00.000Z,2006-08-13T00:00:00.000Z,2006-07-22 00:00:00,3,81.91,Destination JYDLM,Carrera1234 con Ave. Carlos Soublette #8-35,San Cristóbal,Táchira,10199,Venezuela


In [0]:
#checking for nullvalues
null_check = salesorder_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in salesorder_bronze.columns
])

display(null_check)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,0,0,0,0,0,0,0,0,0,0,0,0,0


A small insight here is that the original OLTP source file i saw null values in the ship region col but here its showing 0 null values so there might be some discripency in the dataset

In [0]:

from pyspark.sql.functions import col, count, when, trim
empty_check = salesorder_bronze.select([
    count(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == ""),# checkin if null values are stored as an empty string
            c
        )
    ).alias(c)
    for c in salesorder_bronze.columns
])

display(empty_check)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
salesorder_bronze.select(
    "orderId",
    "orderDate",
    "requiredDate",
    "shippedDate",
    "shipRegion"
).show(20, truncate=False)

+-------+-------------------+-------------------+-------------------+----------+
|orderId|orderDate          |requiredDate       |shippedDate        |shipRegion|
+-------+-------------------+-------------------+-------------------+----------+
|10248  |2006-07-04 00:00:00|2006-08-01 00:00:00|2006-07-16 00:00:00|NULL      |
|10249  |2006-07-05 00:00:00|2006-08-16 00:00:00|2006-07-10 00:00:00|NULL      |
|10250  |2006-07-08 00:00:00|2006-08-05 00:00:00|2006-07-12 00:00:00|RJ        |
|10251  |2006-07-08 00:00:00|2006-08-05 00:00:00|2006-07-15 00:00:00|NULL      |
|10252  |2006-07-09 00:00:00|2006-08-06 00:00:00|2006-07-11 00:00:00|NULL      |
|10253  |2006-07-10 00:00:00|2006-07-24 00:00:00|2006-07-16 00:00:00|RJ        |
|10254  |2006-07-11 00:00:00|2006-08-08 00:00:00|2006-07-23 00:00:00|NULL      |
|10255  |2006-07-12 00:00:00|2006-08-09 00:00:00|2006-07-15 00:00:00|NULL      |
|10256  |2006-07-15 00:00:00|2006-08-12 00:00:00|2006-07-17 00:00:00|SP        |
|10257  |2006-07-16 00:00:00

#### there are definietly null values in the ship region col

In [0]:
salesorder_bronze.filter(
    col("shipRegion").isNull()
).count()

0

In [0]:
empty_check = salesorder_bronze.select([
    count(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == "NULL"),# checkin if null values are stored as "NULL"
            c
        )
    ).alias(c)
    for c in salesorder_bronze.columns
])

display(empty_check)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,0,0,0,0,21,0,0,0,0,0,507,0,0


In [0]:
salesorder_bronze.filter(
    col("shipRegion").isNull()
).count()

0

The shippedDate column contains 21 literal "NULL" strings, while shipRegion contains 507 literal "NULL" strings.
We’ll convert these "NULL" strings into proper Spark NULL values in the Silver layer.

In [0]:
#Standardizing the data and converting the string "NULL" to actual null values
salesorder_silver = salesorder_bronze.select(
    col("orderId").alias("order_id"),
    col("custId").alias("customer_id"),
    col("employeeId").alias("employee_id"),
    col("orderDate").alias("order_date"),
    col("requiredDate").alias("required_date"),
    when(col("shippedDate") == "NULL", None)
        .otherwise(col("shippedDate")).alias("shipped_date"),
    col("shipperid").alias("shipper_id"),
    col("freight"),
    col("shipName").alias("ship_name"),
    col("shipAddress").alias("ship_address"),
    col("shipCity").alias("ship_city"),
    when(col("shipRegion") == "NULL", None)
        .otherwise(col("shipRegion")).alias("ship_region"),
    col("shipPostalCode").alias("ship_postal_code"),
    col("shipCountry").alias("ship_country")
)

display(salesorder_silver)

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
10248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France
10249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,null,10328,Germany
10250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
10251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,null,10342,France
10252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,null,10318,Belgium
10253,34,3,2006-07-10T00:00:00.000Z,2006-07-24T00:00:00.000Z,2006-07-16 00:00:00,2,58.17,Destination JPAIY,"Rua do Paço, 8901",Rio de Janeiro,RJ,10196,Brazil
10254,14,5,2006-07-11T00:00:00.000Z,2006-08-08T00:00:00.000Z,2006-07-23 00:00:00,2,22.98,Destination YUJRD,Hauptstr. 1234,Bern,null,10139,Switzerland
10255,68,9,2006-07-12T00:00:00.000Z,2006-08-09T00:00:00.000Z,2006-07-15 00:00:00,3,148.33,Ship to 68-A,Starenweg 6789,Genève,null,10294,Switzerland
10256,88,3,2006-07-15T00:00:00.000Z,2006-08-12T00:00:00.000Z,2006-07-17 00:00:00,2,13.97,Ship to 88-B,"Rua do Mercado, 5678",Resende,SP,10354,Brazil
10257,35,4,2006-07-16T00:00:00.000Z,2006-08-13T00:00:00.000Z,2006-07-22 00:00:00,3,81.91,Destination JYDLM,Carrera1234 con Ave. Carlos Soublette #8-35,San Cristóbal,Táchira,10199,Venezuela


In [0]:
#checking
salesorder_silver.filter(
    col("shipped_date").isNull()
).count()

21

In [0]:
salesorder_silver.filter(
    col("ship_region").isNull()
).count()

507

In [0]:
#saving the silver table 
salesorder_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.silver_salesorder")

In [0]:
display(spark.table("workspace.northwind.silver_salesorder"))

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
10248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France
10249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,null,10328,Germany
10250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
10251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,null,10342,France
10252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,null,10318,Belgium
10253,34,3,2006-07-10T00:00:00.000Z,2006-07-24T00:00:00.000Z,2006-07-16 00:00:00,2,58.17,Destination JPAIY,"Rua do Paço, 8901",Rio de Janeiro,RJ,10196,Brazil
10254,14,5,2006-07-11T00:00:00.000Z,2006-08-08T00:00:00.000Z,2006-07-23 00:00:00,2,22.98,Destination YUJRD,Hauptstr. 1234,Bern,null,10139,Switzerland
10255,68,9,2006-07-12T00:00:00.000Z,2006-08-09T00:00:00.000Z,2006-07-15 00:00:00,3,148.33,Ship to 68-A,Starenweg 6789,Genève,null,10294,Switzerland
10256,88,3,2006-07-15T00:00:00.000Z,2006-08-12T00:00:00.000Z,2006-07-17 00:00:00,2,13.97,Ship to 88-B,"Rua do Mercado, 5678",Resende,SP,10354,Brazil
10257,35,4,2006-07-16T00:00:00.000Z,2006-08-13T00:00:00.000Z,2006-07-22 00:00:00,3,81.91,Destination JYDLM,Carrera1234 con Ave. Carlos Soublette #8-35,San Cristóbal,Táchira,10199,Venezuela


### Silver table :- orderdetails

In [0]:
orderdetail_bronze = spark.table("workspace.northwind.bronze_orderdetail")

orderdetail_bronze.printSchema()

display(orderdetail_bronze)

root
 |-- orderDetailId: integer (nullable = true)
 |-- orderId: integer (nullable = true)
 |-- productId: integer (nullable = true)
 |-- unitPrice: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



orderDetailId,orderId,productId,unitPrice,quantity,discount
1,10248,11,14.0,12,0.0
2,10248,42,9.8,10,0.0
3,10248,72,34.8,5,0.0
4,10249,14,18.6,9,0.0
5,10249,51,42.4,40,0.0
6,10250,41,7.7,10,0.0
7,10250,51,42.4,35,0.15
8,10250,65,16.8,15,0.15
9,10251,22,16.8,6,0.05
10,10251,57,15.6,15,0.05


In [0]:
#checking null 
null_check = orderdetail_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orderdetail_bronze.columns
])

display(null_check)

orderDetailId,orderId,productId,unitPrice,quantity,discount
0,0,0,0,0,0


In [0]:
#chceking duplicates
duplicate_orderdetails = (
    orderdetail_bronze
    .groupBy("orderDetailId")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_orderdetails)

orderDetailId,count


In [0]:
#checking the same 
null_string_check = orderdetail_bronze.select([
    count(
        when(
            trim(col(c).cast("string")) == "NULL",
            c
        )
    ).alias(c)
    for c in orderdetail_bronze.columns
])

display(null_string_check)

orderDetailId,orderId,productId,unitPrice,quantity,discount
0,0,0,0,0,0


In [0]:
#giving the col proper names
orderdetail_silver = orderdetail_bronze.select(
    col("orderDetailId").alias("order_detail_id"),
    col("orderId").alias("order_id"),
    col("productId").alias("product_id"),
    col("unitPrice").alias("unit_price"),
    col("quantity"),
    col("discount")
)

display(orderdetail_silver)

order_detail_id,order_id,product_id,unit_price,quantity,discount
1,10248,11,14.0,12,0.0
2,10248,42,9.8,10,0.0
3,10248,72,34.8,5,0.0
4,10249,14,18.6,9,0.0
5,10249,51,42.4,40,0.0
6,10250,41,7.7,10,0.0
7,10250,51,42.4,35,0.15
8,10250,65,16.8,15,0.15
9,10251,22,16.8,6,0.05
10,10251,57,15.6,15,0.05


In [0]:
#saving the silver table
orderdetail_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.silver_orderdetail")

### Silver table :- product 

In [0]:
product_bronze = spark.table("workspace.northwind.bronze_product")

product_bronze.printSchema()

display(product_bronze)

root
 |-- productId: integer (nullable = true)
 |-- productName: string (nullable = true)
 |-- supplierId: integer (nullable = true)
 |-- categoryId: integer (nullable = true)
 |-- quantityPerUnit: string (nullable = true)
 |-- unitPrice: double (nullable = true)
 |-- unitsInStock: string (nullable = true)
 |-- unitsOnOrder: string (nullable = true)
 |-- reorderLevel: string (nullable = true)
 |-- discontinued: integer (nullable = true)



productId,productName,supplierId,categoryId,quantityPerUnit,unitPrice,unitsInStock,unitsOnOrder,reorderLevel,discontinued
1,Product HHYDP,1,1,NULL,18.0,NULL,NULL,NULL,0
2,Product RECZE,1,1,NULL,19.0,NULL,NULL,NULL,0
3,Product IMEHJ,1,2,NULL,10.0,NULL,NULL,NULL,0
4,Product KSBRM,2,2,NULL,22.0,NULL,NULL,NULL,0
5,Product EPEIM,2,2,NULL,21.35,NULL,NULL,NULL,1
6,Product VAIIV,3,2,NULL,25.0,NULL,NULL,NULL,0
7,Product HMLNI,3,7,NULL,30.0,NULL,NULL,NULL,0
8,Product WVJFP,3,2,NULL,40.0,NULL,NULL,NULL,0
9,Product AOZBW,4,6,NULL,97.0,NULL,NULL,NULL,1
10,Product YHXGE,4,8,NULL,31.0,NULL,NULL,NULL,0


In [0]:
#chceking null 
null_check = product_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in product_bronze.columns
])

display(null_check)

productId,productName,supplierId,categoryId,quantityPerUnit,unitPrice,unitsInStock,unitsOnOrder,reorderLevel,discontinued
0,0,0,0,0,0,0,0,0,0


In [0]:
#duplictaes
duplicate_products = (
    product_bronze
    .groupBy("productId")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_products)

productId,count


In [0]:
null_string_check = product_bronze.select([
    count(
        when(
            trim(col(c).cast("string")) == "NULL",
            c
        )
    ).alias(c)
    for c in product_bronze.columns
])

display(null_string_check)

productId,productName,supplierId,categoryId,quantityPerUnit,unitPrice,unitsInStock,unitsOnOrder,reorderLevel,discontinued
0,0,0,0,77,0,77,77,77,0


In [0]:
product_bronze.select(
    "productId",
    "quantityPerUnit",
    "unitsInStock",
    "unitsOnOrder",
    "reorderLevel"
).show(100, truncate=False) #prevents Spark from shortening long text values

+---------+---------------+------------+------------+------------+
|productId|quantityPerUnit|unitsInStock|unitsOnOrder|reorderLevel|
+---------+---------------+------------+------------+------------+
|1        |NULL           |NULL        |NULL        |NULL        |
|2        |NULL           |NULL        |NULL        |NULL        |
|3        |NULL           |NULL        |NULL        |NULL        |
|4        |NULL           |NULL        |NULL        |NULL        |
|5        |NULL           |NULL        |NULL        |NULL        |
|6        |NULL           |NULL        |NULL        |NULL        |
|7        |NULL           |NULL        |NULL        |NULL        |
|8        |NULL           |NULL        |NULL        |NULL        |
|9        |NULL           |NULL        |NULL        |NULL        |
|10       |NULL           |NULL        |NULL        |NULL        |
|11       |NULL           |NULL        |NULL        |NULL        |
|12       |NULL           |NULL        |NULL        |NULL     

4 columns have no values in them ie all the values are null values

In [0]:
display(
    spark.table("workspace.northwind.bronze_product")
    .select(
        "productId",
        "productName",
        "quantityPerUnit",
        "unitsInStock",
        "unitsOnOrder",
        "reorderLevel"
    )
)

productId,productName,quantityPerUnit,unitsInStock,unitsOnOrder,reorderLevel
1,Product HHYDP,NULL,NULL,NULL,NULL
2,Product RECZE,NULL,NULL,NULL,NULL
3,Product IMEHJ,NULL,NULL,NULL,NULL
4,Product KSBRM,NULL,NULL,NULL,NULL
5,Product EPEIM,NULL,NULL,NULL,NULL
6,Product VAIIV,NULL,NULL,NULL,NULL
7,Product HMLNI,NULL,NULL,NULL,NULL
8,Product WVJFP,NULL,NULL,NULL,NULL
9,Product AOZBW,NULL,NULL,NULL,NULL
10,Product YHXGE,NULL,NULL,NULL,NULL


### Insights:-
The product table has 77 NULL values in quantityPerUnit, unitsInStock, unitsOnOrder, and reorderLevel  and these NULLs are present in the original source, we'll preserve them in Silver rather than treating them as bad data.

In [0]:
#treating the "NULL "

product_silver = product_bronze.select(
    col("productId").alias("product_id"),
    col("productName").alias("product_name"),
    col("supplierId").alias("supplier_id"),
    col("categoryId").alias("category_id"),
    when(col("quantityPerUnit") == "NULL", None)
        .otherwise(col("quantityPerUnit"))
        .alias("quantity_per_unit"),
    col("unitPrice").alias("unit_price"),
    when(col("unitsInStock") == "NULL", None)
        .otherwise(col("unitsInStock"))
        .cast("int")
        .alias("units_in_stock"),
    when(col("unitsOnOrder") == "NULL", None)
        .otherwise(col("unitsOnOrder"))
        .cast("int")
        .alias("units_on_order"),
    when(col("reorderLevel") == "NULL", None)
        .otherwise(col("reorderLevel"))
        .cast("int")
        .alias("reorder_level"),
    col("discontinued")
)

display(product_silver)

product_id,product_name,supplier_id,category_id,quantity_per_unit,unit_price,units_in_stock,units_on_order,reorder_level,discontinued
1,Product HHYDP,1,1,null,18.0,null,null,null,0
2,Product RECZE,1,1,null,19.0,null,null,null,0
3,Product IMEHJ,1,2,null,10.0,null,null,null,0
4,Product KSBRM,2,2,null,22.0,null,null,null,0
5,Product EPEIM,2,2,null,21.35,null,null,null,1
6,Product VAIIV,3,2,null,25.0,null,null,null,0
7,Product HMLNI,3,7,null,30.0,null,null,null,0
8,Product WVJFP,3,2,null,40.0,null,null,null,0
9,Product AOZBW,4,6,null,97.0,null,null,null,1
10,Product YHXGE,4,8,null,31.0,null,null,null,0


In [0]:
product_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.silver_product")

### Silver table :- category 


In [0]:
category_bronze = spark.table("workspace.northwind.bronze_category")

category_bronze.printSchema()

display(category_bronze)

root
 |-- categoryId: integer (nullable = true)
 |-- categoryName: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)



categoryId,categoryName,description,picture
1,Beverages,"Soft drinks, coffees, teas, beers, and ales",NULL
2,Condiments,"Sweet and savory sauces, relishes, spreads, and seasonings",NULL
3,Confections,"Desserts, candies, and sweet breads",NULL
4,Dairy Products,Cheeses,NULL
5,Grains/Cereals,"Breads, crackers, pasta, and cereal",NULL
6,Meat/Poultry,Prepared meats,NULL
7,Produce,Dried fruit and bean curd,NULL
8,Seafood,Seaweed and fish,NULL


In [0]:
#chceking NULL
null_check = category_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in category_bronze.columns
])

display(null_check)

categoryId,categoryName,description,picture
0,0,0,0


In [0]:
#duplictaes
duplicate_categories = (
    category_bronze
    .groupBy("categoryId")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_categories)

categoryId,count


In [0]:
null_string_check = category_bronze.select([
    count(
        when(
            trim(col(c).cast("string")) == "NULL",
            c
        )
    ).alias(c)
    for c in category_bronze.columns
])

display(null_string_check)

categoryId,categoryName,description,picture
0,0,0,8


In [0]:
#treating the "NULL "
from pyspark.sql.functions import col, when

category_silver = category_bronze.select(
    col("categoryId").alias("category_id"),
    col("categoryName").alias("category_name"),
    col("description"),
    when(col("picture") == "NULL", None)
        .otherwise(col("picture"))
        .alias("picture")
)

display(category_silver)

category_id,category_name,description,picture
1,Beverages,"Soft drinks, coffees, teas, beers, and ales",null
2,Condiments,"Sweet and savory sauces, relishes, spreads, and seasonings",null
3,Confections,"Desserts, candies, and sweet breads",null
4,Dairy Products,Cheeses,null
5,Grains/Cereals,"Breads, crackers, pasta, and cereal",null
6,Meat/Poultry,Prepared meats,null
7,Produce,Dried fruit and bean curd,null
8,Seafood,Seaweed and fish,null


In [0]:
#saving 
category_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.northwind.silver_category")

### ALL 5 Silver tables have been created we are done with the silver layer

In [0]:
customer_silver.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- mobile: string (nullable = true)
 |-- email: string (nullable = true)
 |-- fax: string (nullable = true)



### creating Dimension and fact tables

In [0]:
#creating cust dimension table
from pyspark.sql.functions import monotonically_increasing_id #generates a new unique-looking numeric ID for each customer.

dim_customer = (
    customer_silver
    .withColumn("customer_key", monotonically_increasing_id())
    .select(
        "customer_key",  #this will become our surrogate key 
        "customer_id",
        "company_name",
        "contact_name",
        "contact_title",
        "address",
        "city",
        "region",
        "postal_code",
        "country",
        "phone",
        "mobile",
        "email",
        "fax"
    )
)
dim_customer.show(5)

+------------+-----------+--------------+-----------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+------+-----+--------------+
|customer_key|customer_id|  company_name|     contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|mobile|email|           fax|
+------------+-----------+--------------+-----------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+------+-----+--------------+
|           0|          1|Customer NRZBB|   Allen, Michael|Sales Representative|     Obere Str. 0123|     Berlin|  NULL|      10092|Germany|   030-3456789|  NULL| NULL|   030-0123456|
|           1|          2|Customer MLTDN|    Hassall, Mark|               Owner|Avda. de la Const...|México D.F.|  NULL|      10077| Mexico|  (5) 789-0123|  NULL| NULL|  (5) 456-7890|
|           2|          3|Customer KBUDE|    Peoples, John|               Owner|

In [0]:
dim_customer.display()

customer_key,customer_id,company_name,contact_name,contact_title,address,city,region,postal_code,country,phone,mobile,email,fax
0,1,Customer NRZBB,"Allen, Michael",Sales Representative,Obere Str. 0123,Berlin,NULL,10092,Germany,030-3456789,NULL,NULL,030-0123456
1,2,Customer MLTDN,"Hassall, Mark",Owner,Avda. de la Constitución 5678,México D.F.,NULL,10077,Mexico,(5) 789-0123,NULL,NULL,(5) 456-7890
2,3,Customer KBUDE,"Peoples, John",Owner,Mataderos 7890,México D.F.,NULL,10097,Mexico,(5) 123-4567,NULL,NULL,NULL
3,4,Customer HFBZG,"Arndt, Torsten",Sales Representative,7890 Hanover Sq.,London,NULL,10046,UK,(171) 456-7890,NULL,NULL,(171) 456-7891
4,5,Customer HGVLZ,"Higginbotham, Tom",Order Administrator,Berguvsvägen 5678,Luleå,NULL,10112,Sweden,0921-67 89 01,NULL,NULL,0921-23 45 67
5,6,Customer XHXJV,"Poland, Carole",Sales Representative,Forsterstr. 7890,Mannheim,NULL,10117,Germany,0621-67890,NULL,NULL,0621-12345
6,7,Customer QXVLA,"Bansal, Dushyant",Marketing Manager,"2345, place Kléber",Strasbourg,NULL,10089,France,67.89.01.23,NULL,NULL,67.89.01.24
7,8,Customer QUHWH,"Ilyina, Julia",Owner,"C/ Araquil, 0123",Madrid,NULL,10104,Spain,(91) 345 67 89,NULL,NULL,(91) 012 34 56
8,9,Customer RTXGC,"Raghav, Amritansh",Owner,"6789, rue des Bouchers",Marseille,NULL,10105,France,23.45.67.89,NULL,NULL,23.45.67.80
9,10,Customer EEALV,"Bassols, Pilar Colome",Accounting Manager,8901 Tsawassen Blvd.,Tsawassen,BC,10111,Canada,(604) 901-2345,NULL,NULL,(604) 678-9012


In [0]:
dim_customer.printSchema()

root
 |-- customer_key: long (nullable = false)
 |-- customer_id: integer (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- mobile: string (nullable = true)
 |-- email: string (nullable = true)
 |-- fax: string (nullable = true)



### surrogate key
i created a surrogate key because the warehouse should have its own unique identifier for each dimension record, it helps preserve the data in the DW even if there is a change in the source table we can still identify the records with the help of surrogate key

In [0]:
#saving it 
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_customer")

### CReating Dimension table for the product table

In [0]:
product_silver.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- quantity_per_unit: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_in_stock: integer (nullable = true)
 |-- units_on_order: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- discontinued: integer (nullable = true)



In [0]:
#creating the surrogate key & DIm_product
dim_product = (
    product_silver
    .withColumn("product_key", monotonically_increasing_id())
    .select(
        "product_key",
        "product_id",
        "product_name",
        "supplier_id",
        "category_id",
        "quantity_per_unit",
        "unit_price",
        "units_in_stock",
        "units_on_order",
        "reorder_level",
        "discontinued"
    )
)

In [0]:
dim_product.show(5)

+-----------+----------+-------------+-----------+-----------+-----------------+----------+--------------+--------------+-------------+------------+
|product_key|product_id| product_name|supplier_id|category_id|quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|
+-----------+----------+-------------+-----------+-----------+-----------------+----------+--------------+--------------+-------------+------------+
|          0|         1|Product HHYDP|          1|          1|             NULL|      18.0|          NULL|          NULL|         NULL|           0|
|          1|         2|Product RECZE|          1|          1|             NULL|      19.0|          NULL|          NULL|         NULL|           0|
|          2|         3|Product IMEHJ|          1|          2|             NULL|      10.0|          NULL|          NULL|         NULL|           0|
|          3|         4|Product KSBRM|          2|          2|             NULL|      22.0|          NULL|

In [0]:
#savng it 
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_product")

### Dimension table :- Category

In [0]:
category_silver.printSchema()

root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)



In [0]:
dim_category = (
    category_silver
    .withColumn("category_key", monotonically_increasing_id())
    .select(
        "category_key",
        "category_id",
        "category_name",
        "description",
        "picture"
    )
)

In [0]:
dim_category.show(5)

+------------+-----------+--------------+--------------------+-------+
|category_key|category_id| category_name|         description|picture|
+------------+-----------+--------------+--------------------+-------+
|           0|          1|     Beverages|Soft drinks, coff...|   NULL|
|           1|          2|    Condiments|Sweet and savory ...|   NULL|
|           2|          3|   Confections|Desserts, candies...|   NULL|
|           3|          4|Dairy Products|             Cheeses|   NULL|
|           4|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+------------+-----------+--------------+--------------------+-------+
only showing top 5 rows


In [0]:
#saving it 
dim_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_category")

### we have created all the dimension tables now lets move to the fact table

In [0]:
salesorder_silver.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- required_date: timestamp (nullable = true)
 |-- shipped_date: string (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: integer (nullable = true)
 |-- ship_country: string (nullable = true)



In [0]:
orderdetail_silver.printSchema()

root
 |-- order_detail_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



#### Defining the Fact Table Grain (1 Row = 1 Product Line per Order) and Joining Order & Order-Detail Tables

In [0]:
sales_detail = (
    orderdetail_silver
    .join(
        salesorder_silver,
        on="order_id",
        how="inner"
    )
)

In [0]:
sales_detail.show(5)

+--------+---------------+----------+----------+--------+--------+-----------+-----------+-------------------+-------------------+-------------------+----------+-------+-----------------+--------------------+--------------+-----------+----------------+------------+
|order_id|order_detail_id|product_id|unit_price|quantity|discount|customer_id|employee_id|         order_date|      required_date|       shipped_date|shipper_id|freight|        ship_name|        ship_address|     ship_city|ship_region|ship_postal_code|ship_country|
+--------+---------------+----------+----------+--------+--------+-----------+-----------+-------------------+-------------------+-------------------+----------+-------+-----------------+--------------------+--------------+-----------+----------------+------------+
|   10248|              3|        72|      34.8|       5|     0.0|         85|          5|2006-07-04 00:00:00|2006-08-01 00:00:00|2006-07-16 00:00:00|         3|  32.38|     Ship to 85-B|6789 rue de l'A

### joing the fact table with each of the  dimension table :- 1. fact to cust_dim
We have Dim_cust(dimension table for customers) and sales_detail(our fact table ) now lets join them

In [0]:
sales_detail_customer = (
    sales_detail
    .join(
        dim_customer.select("customer_id", "customer_key"),
        on="customer_id",
        how="left"    #left join cause we want all the records from the sales table even if for some reason a matching customer isn't found in the dimension.
    )
)

In [0]:
sales_detail_customer.select(
    "order_id",
    "customer_id",
    "customer_key"
).show(5)

+--------+-----------+------------+
|order_id|customer_id|customer_key|
+--------+-----------+------------+
|   10248|         85|          84|
|   10248|         85|          84|
|   10248|         85|          84|
|   10249|         79|          78|
|   10249|         79|          78|
+--------+-----------+------------+
only showing top 5 rows


### JOining the new created table sales_detail_product with dim product 
Our sales data contains product_id, while dim_product contains product_id and product_key. We join them so each sale can be associated with the product's warehouse surrogate key

In [0]:
#now lets join product id with product key 
sales_detail_product = (
    sales_detail_customer
    .join(
        dim_product.select("product_id", "product_key"),
        on="product_id",
        how="left"
    )
)

In [0]:
sales_detail_product.select(
    "order_id",
    "product_id",
    "product_key"
).show(5)

+--------+----------+-----------+
|order_id|product_id|product_key|
+--------+----------+-----------+
|   10248|        11|         10|
|   10248|        42|         41|
|   10248|        72|         71|
|   10249|        14|         13|
|   10249|        51|         50|
+--------+----------+-----------+
only showing top 5 rows


### Joining sales_detail_product + dim_product + dim_category
We need the category associated with each product. dim_product provides the category_id, and dim_category maps that category_id to the warehouse's category_key. This allows our fact table to identify which category each sale belongs to.

In [0]:
sales_detail_category = (
    sales_detail_product
    .join(
        dim_product.select("product_id", "category_id"),
        on="product_id",
        how="left"
    )
    .join(
        dim_category.select("category_id", "category_key"),
        on="category_id",
        how="left"
    )
)

In [0]:
sales_detail_category.select(
    "product_id",
    "category_id",
    "category_key"
).show(5)

+----------+-----------+------------+
|product_id|category_id|category_key|
+----------+-----------+------------+
|        11|          4|           3|
|        42|          5|           4|
|        72|          4|           3|
|        14|          7|           6|
|        51|          7|           6|
+----------+-----------+------------+
only showing top 5 rows


###CReating the actual fact sales col
here we will only add those columns that are required in th efact table

In [0]:
from pyspark.sql.functions import col, round

fact_sales = (
    sales_detail_category
    .withColumn(
        "sales_amount",
        round(
            col("unit_price") * col("quantity") * (1 - col("discount")),
            2
        )
    )
    .select(
        "order_detail_id",
        "order_id",
        "customer_key",
        "product_key",
        "category_key",
        "order_date",
        "quantity",
        "unit_price",
        "discount",
        "sales_amount",
        "freight"
    )
)

fact_sales.show(10)

+---------------+--------+------------+-----------+------------+-------------------+--------+----------+--------+------------+-------+
|order_detail_id|order_id|customer_key|product_key|category_key|         order_date|quantity|unit_price|discount|sales_amount|freight|
+---------------+--------+------------+-----------+------------+-------------------+--------+----------+--------+------------+-------+
|              1|   10248|          84|         10|           3|2006-07-04 00:00:00|      12|      14.0|     0.0|       168.0|  32.38|
|              2|   10248|          84|         41|           4|2006-07-04 00:00:00|      10|       9.8|     0.0|        98.0|  32.38|
|              3|   10248|          84|         71|           3|2006-07-04 00:00:00|       5|      34.8|     0.0|       174.0|  32.38|
|              4|   10249|          78|         13|           6|2006-07-05 00:00:00|       9|      18.6|     0.0|       167.4|  11.61|
|              5|   10249|          78|         50|    

In [0]:
#checkinig our fact table 
fact_sales.count()

2155

In [0]:
fact_sales.select("order_id", "order_detail_id").distinct().count() #chcking whether our join created any duplicate rows

2155

In [0]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales")

spark.table("fact_sales").show(10)

+---------------+--------+------------+-----------+------------+-------------------+--------+----------+--------+------------+-------+
|order_detail_id|order_id|customer_key|product_key|category_key|         order_date|quantity|unit_price|discount|sales_amount|freight|
+---------------+--------+------------+-----------+------------+-------------------+--------+----------+--------+------------+-------+
|              1|   10248|          84|         10|           3|2006-07-04 00:00:00|      12|      14.0|     0.0|       168.0|  32.38|
|              2|   10248|          84|         41|           4|2006-07-04 00:00:00|      10|       9.8|     0.0|        98.0|  32.38|
|              3|   10248|          84|         71|           3|2006-07-04 00:00:00|       5|      34.8|     0.0|       174.0|  32.38|
|              4|   10249|          78|         13|           6|2006-07-05 00:00:00|       9|      18.6|     0.0|       167.4|  11.61|
|              5|   10249|          78|         50|    

### We completed the Gold layer by creating a star schema with fact_sales at the center and three dimension tables: dim_customer, dim_product, and dim_category.
We connected the fact table to the dimensions using surrogate keys and calculated the sales_amount business measure.

### Next step :-
we have created the golds layer now lets find some insight out of the golds table

### Total sale amount 


In [0]:
#Total sales
from pyspark.sql.functions import sum

total_sales = (
    fact_sales
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
)

total_sales.show()

+-----------+
|total_sales|
+-----------+
| 1265793.25|
+-----------+



### TOTAL ORDERS


In [0]:
#total orders
from pyspark.sql.functions import countDistinct, round

total_orders = (
    fact_sales
    .agg(
        round(countDistinct("order_id"), 2).alias("total_orders")
    )
)

total_orders.show()

+------------+
|total_orders|
+------------+
|         830|
+------------+



### AVg order vale

In [0]:
from pyspark.sql.functions import sum, countDistinct, round

aov = (
    fact_sales
    .agg(
        round(
            sum("sales_amount") / countDistinct("order_id"),
            2
        ).alias("average_order_value")
    )
)

aov.show()

+-------------------+
|average_order_value|
+-------------------+
|            1525.05|
+-------------------+



### 4.Total Quantity Sold

In [0]:
from pyspark.sql.functions import sum

total_quantity = (
    fact_sales
    .agg(
        sum("quantity").alias("total_quantity_sold")
    )
)

total_quantity.show()

+-------------------+
|total_quantity_sold|
+-------------------+
|              51317|
+-------------------+



### 5. Average Quantity per Order

In [0]:
from pyspark.sql.functions import sum, countDistinct, round

avg_quantity_per_order = (
    fact_sales
    .agg(
        round(
            sum("quantity") / countDistinct("order_id"),
            2
        ).alias("avg_quantity_per_order")
    )
)

avg_quantity_per_order.show()

+----------------------+
|avg_quantity_per_order|
+----------------------+
|                 61.83|
+----------------------+



### 6. Total Customers

In [0]:
from pyspark.sql.functions import countDistinct

total_customers = (
    fact_sales
    .agg(
        countDistinct("customer_key").alias("total_customers")
    )
)

total_customers.show()

+---------------+
|total_customers|
+---------------+
|             89|
+---------------+



### 7. Average Sales per Customer

In [0]:
from pyspark.sql.functions import sum, countDistinct, round

avg_sales_per_customer = (
    fact_sales
    .agg(
        round(
            sum("sales_amount") / countDistinct("customer_key"),
            2
        ).alias("avg_sales_per_customer")
    )
)

avg_sales_per_customer.show()

+----------------------+
|avg_sales_per_customer|
+----------------------+
|               14222.4|
+----------------------+



### 8. Total Sales by Category

In [0]:
from pyspark.sql.functions import sum, round

sales_by_category = (
    fact_sales
    .groupBy("category_key")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("total_sales", ascending=False)
)

sales_by_category.show()

+------------+-----------+
|category_key|total_sales|
+------------+-----------+
|           0|   267868.2|
|           3|  234507.32|
|           2|  167357.26|
|           5|  163022.38|
|           7|  131261.76|
|           1|  106047.15|
|           6|   99984.58|
|           4|    95744.6|
+------------+-----------+



### IMP:-
The fact table uses surrogate keys to maintain relationships with dimension tables. For reporting, I join the fact with the dimension to retrieve descriptive attributes such as category_name. This keeps the fact table optimized while the dimension provides the business-readable information.

In [0]:
#lets get the category name as well by joining fact sales with category dimension table
from pyspark.sql.functions import sum, round

sales_by_category = (
    fact_sales
    .join(
        dim_category.select("category_key", "category_name"),
        on="category_key",
        how="left"
    )
    .groupBy("category_key", "category_name")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("total_sales", ascending=False)
)

sales_by_category.show()


+------------+--------------+-----------+
|category_key| category_name|total_sales|
+------------+--------------+-----------+
|           0|     Beverages|   267868.2|
|           3|Dairy Products|  234507.32|
|           2|   Confections|  167357.26|
|           5|  Meat/Poultry|  163022.38|
|           7|       Seafood|  131261.76|
|           1|    Condiments|  106047.15|
|           6|       Produce|   99984.58|
|           4|Grains/Cereals|    95744.6|
+------------+--------------+-----------+



### 9. Top 10 Products by Sales
for this we will  join fact_sales with product dimension so we can use the actual product name instead of just product_key

In [0]:
from pyspark.sql.functions import sum

top_products = (
    fact_sales
    .join(
        dim_product.select("product_key", "product_name"),
        on="product_key",
        how="left"
    )
    .groupBy("product_key", "product_name")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("total_sales", ascending=False)
    .limit(10)
)

top_products.show()

+-----------+-------------+-----------+
|product_key| product_name|total_sales|
+-----------+-------------+-----------+
|         37|Product QDOMO|  141396.74|
|         28|Product VJXYN|   80368.69|
|         58|Product UKXRI|    71155.7|
|         61|Product WUXYK|   47234.98|
|         59|Product WHBYK|   46825.48|
|         55|Product VKCMF|   42593.06|
|         50|Product APITJ|   41819.65|
|         16|Product BLCAX|   32698.38|
|         17|Product CKEDC|   29171.88|
|         27|Product OFBNT|   25696.64|
+-----------+-------------+-----------+



### 10. Monthly Sales Trend

In [0]:
from pyspark.sql.functions import sum, round, date_format

monthly_sales = (
    fact_sales
    .withColumn(
        "month",
        date_format("order_date", "MMMM yyyy")
    )
    .groupBy("month")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("month")
)

monthly_sales.show(30)

+--------------+-----------+
|         month|total_sales|
+--------------+-----------+
|    April 2007|   53032.95|
|    April 2008|  123798.69|
|   August 2006|   25485.28|
|   August 2007|   47287.68|
| December 2006|   45239.63|
| December 2007|   71398.44|
| February 2007|   38483.64|
| February 2008|   99415.29|
|  January 2007|   61258.08|
|  January 2008|   94222.13|
|     July 2006|    27861.9|
|     July 2007|   51020.88|
|     June 2007|   36362.82|
|    March 2007|   38547.23|
|    March 2008|  104854.19|
|      May 2007|    53781.3|
|      May 2008|   18333.62|
| November 2006|   45600.05|
| November 2007|   43533.81|
|  October 2006|   37515.73|
|  October 2007|   66749.24|
|September 2006|    26381.4|
|September 2007|   55629.27|
+--------------+-----------+



In [0]:
#getting the top 10 sales by month

top_months = (
    fact_sales
    .withColumn(
        "month",
        date_format("order_date", "MMMM yyyy")
    )
    .groupBy("month")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("total_sales", ascending=False)
    .limit(10)
)

top_months.show()

+--------------+-----------+
|         month|total_sales|
+--------------+-----------+
|    April 2008|  123798.69|
|    March 2008|  104854.19|
| February 2008|   99415.29|
|  January 2008|   94222.13|
| December 2007|   71398.44|
|  October 2007|   66749.24|
|  January 2007|   61258.08|
|September 2007|   55629.27|
|      May 2007|    53781.3|
|    April 2007|   53032.95|
+--------------+-----------+



### Validating the Gold layer

In [0]:
#chcking if all 4 gold table exist 
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|    big_mart_sales|      false|
| default|  bronze_customers|      false|
| default|bronze_order_items|      false|
| default|     bronze_orders|      false|
| default|   bronze_payments|      false|
| default|   bronze_products|      false|
| default|      dim_category|      false|
| default|      dim_customer|      false|
| default|       dim_product|      false|
| default|        fact_sales|      false|
| default|gold_sales_summary|      false|
| default|  silver_customers|      false|
| default|silver_order_items|      false|
| default|     silver_orders|      false|
| default|   silver_payments|      false|
| default|   silver_products|      false|
+--------+------------------+-----------+



In [0]:
#we have all our gold tables now lets chck the row
print("dim_customer:", spark.table("dim_customer").count())
print("dim_product:", spark.table("dim_product").count())
print("dim_category:", spark.table("dim_category").count())
print("fact_sales:", spark.table("fact_sales").count())

dim_customer: 91
dim_product: 77
dim_category: 8
fact_sales: 2155


In [0]:
#checking the surrogate key 
print("Duplicate customer keys:",
      dim_customer.count() - dim_customer.select("customer_key").distinct().count())

print("Duplicate product keys:",
      dim_product.count() - dim_product.select("product_key").distinct().count())

print("Duplicate category keys:",
      dim_category.count() - dim_category.select("category_key").distinct().count())

Duplicate customer keys: 0
Duplicate product keys: 0
Duplicate category keys: 0


### CHecking whether every key in fact_sales actually exists in its corresponding dimension.

In [0]:
print("Missing customer keys:",
      fact_sales.join(
          dim_customer.select("customer_key"),
          on="customer_key",
          how="left_anti"
      ).count())

print("Missing product keys:",
      fact_sales.join(
          dim_product.select("product_key"),
          on="product_key",
          how="left_anti"
      ).count())

print("Missing category keys:",
      fact_sales.join(
          dim_category.select("category_key"),
          on="category_key",
          how="left_anti"
      ).count())

Missing customer keys: 0
Missing product keys: 0
Missing category keys: 0


verifying the fact grain 

1 row = 1 product line within an order.

because each row in order_detail_id represents one product line and therefore the amnt of rows should be the same

In [0]:
print(
    "Fact rows:",
    fact_sales.count()
)

print(
    "Distinct order-detail rows:",
    fact_sales.select("order_id", "order_detail_id").distinct().count()
)

Fact rows: 2155
Distinct order-detail rows: 2155


Our Gold star schema is structurally valid, the dimension keys are unique, fact-to-dimension relationships are valid, and the fact-table grain has been preserved at 1 row per order line.

### Incremental Loading :- 1. Sales order

In [0]:
salesorder_path = "/Volumes/workspace/northwind/raw_files/salesorder.csv"
salesorder_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(salesorder_path)
)


In [0]:
display(salesorder_df.limit(10))

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
10248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,NULL,10345,France
10249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,NULL,10328,Germany
10250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
10251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,NULL,10342,France
10252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,NULL,10318,Belgium
10253,34,3,2006-07-10T00:00:00.000Z,2006-07-24T00:00:00.000Z,2006-07-16 00:00:00,2,58.17,Destination JPAIY,"Rua do Paço, 8901",Rio de Janeiro,RJ,10196,Brazil
10254,14,5,2006-07-11T00:00:00.000Z,2006-08-08T00:00:00.000Z,2006-07-23 00:00:00,2,22.98,Destination YUJRD,Hauptstr. 1234,Bern,NULL,10139,Switzerland
10255,68,9,2006-07-12T00:00:00.000Z,2006-08-09T00:00:00.000Z,2006-07-15 00:00:00,3,148.33,Ship to 68-A,Starenweg 6789,Genève,NULL,10294,Switzerland
10256,88,3,2006-07-15T00:00:00.000Z,2006-08-12T00:00:00.000Z,2006-07-17 00:00:00,2,13.97,Ship to 88-B,"Rua do Mercado, 5678",Resende,SP,10354,Brazil
10257,35,4,2006-07-16T00:00:00.000Z,2006-08-13T00:00:00.000Z,2006-07-22 00:00:00,3,81.91,Destination JYDLM,Carrera1234 con Ave. Carlos Soublette #8-35,San Cristóbal,Táchira,10199,Venezuela


In [0]:
#chceking the row count in the original source file 
print("Source CSV row count:", salesorder_df.count())

Source CSV row count: 830


In [0]:
#chceking the silver table row count 
silver_salesorder = spark.table("workspace.northwind.silver_salesorder")

print("Silver row count:", silver_salesorder.count())

Silver row count: 830


In [0]:
silver_salesorder.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- required_date: timestamp (nullable = true)
 |-- shipped_date: string (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: integer (nullable = true)
 |-- ship_country: string (nullable = true)



In [0]:
#verifying that we have 0 new records
new_records = salesorder_df.join(
    silver_salesorder,
    salesorder_df.orderId == silver_salesorder.order_id,  #cols that we are matching
    "left_anti"  #ususally joins are used to combine 2 or more cols but here we are using it to find records that don't exist in Silver
)

print("New records detected:", new_records.count())

New records detected: 0


### UPDATE:- 
i have created a dummy batch/ csv file that contains 5 new records of the sales order table

In [0]:
incremental_path = "/Volumes/workspace/northwind/raw_files/salesorder_incremental_batch.csv"

incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)
incremental_df.printSchema()

root
 |-- orderId: integer (nullable = true)
 |-- custId: integer (nullable = true)
 |-- employeeId: integer (nullable = true)
 |-- orderDate: timestamp (nullable = true)
 |-- requiredDate: timestamp (nullable = true)
 |-- shippedDate: timestamp (nullable = true)
 |-- shipperid: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- shipName: string (nullable = true)
 |-- shipAddress: string (nullable = true)
 |-- shipCity: string (nullable = true)
 |-- shipRegion: string (nullable = true)
 |-- shipPostalCode: integer (nullable = true)
 |-- shipCountry: string (nullable = true)



In [0]:
display(incremental_df)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
20248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16T00:00:00.000Z,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France
20249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10T00:00:00.000Z,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,null,10328,Germany
20250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12T00:00:00.000Z,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
20251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15T00:00:00.000Z,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,null,10342,France
20252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11T00:00:00.000Z,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,null,10318,Belgium


In [0]:
# chceking whether these 5 records wxist in the source file 
new_records = incremental_df.join(
    silver_salesorder,
    incremental_df.orderId == silver_salesorder.order_id,
    "left_anti"
)
print("New records detected:", new_records.count())

New records detected: 5


### Incremental Batch — 3 Cases
1. New record →are completely unique doesn't exist in Silver → simple Insert the record.
2. Existing + identical → duplicate records get added →  ignore it.
3. Existing + changed → Business key already exists but some attributes changed → Update the existing Silver record using MERGE/upsert logic.

Key idea: Don't blindly append the entire incoming batch. First compare it with the existing data using the business key.

In [0]:
#trf the name of the columns of the new records as the same as the col name in silver layer
from pyspark.sql import functions as F

incremental_silver = new_records.select(
    F.col("orderId").alias("order_id"),
    F.col("custId").alias("customer_id"),
    F.col("employeeId").alias("employee_id"),
    F.col("orderDate").alias("order_date"),
    F.col("requiredDate").alias("required_date"),
    F.col("shippedDate").alias("shipped_date"),
    F.col("shipperid").alias("shipper_id"),
    F.col("freight"),
    F.col("shipName").alias("ship_name"),
    F.col("shipAddress").alias("ship_address"),
    F.col("shipCity").alias("ship_city"),
    F.col("shipRegion").alias("ship_region"),
    F.col("shipPostalCode").alias("ship_postal_code"),
    F.col("shipCountry").alias("ship_country")
)

incremental_silver.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- required_date: timestamp (nullable = true)
 |-- shipped_date: timestamp (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: integer (nullable = true)
 |-- ship_country: string (nullable = true)



### Idempotency / duplicate prevention in incremental pipelines :-
in a real world scenario we can get the same incremented file twice this will create duplicates we will counter this prblm now 

In [0]:
new_records = incremental_df.join(
    silver_salesorder,
    incremental_df.orderId == silver_salesorder.order_id,
    "left_anti"
)
print("New records detected:", new_records.count())

New records detected: 5


### Next step:-
add the new records to the silver layer

In [0]:
incremental_silver.write \
    .mode("append") \
    .saveAsTable("workspace.northwind.silver_salesorder")

In [0]:
#verifying
silver_salesorder_updated = spark.table(
    "workspace.northwind.silver_salesorder"
)

print("Silver row count:", silver_salesorder_updated.count())

Silver row count: 835


### Another scenario :-
where we get the same order id but with a different data in it this coulld be a case of data updation

In [0]:
display(incremental_silver)

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country


In [0]:
#creating a dummy record with different freight value but the same order
updated_batch = (
    incremental_df
    .filter(F.col("orderId") == 20248)
    .withColumn("freight", F.lit(99.99))
)
display(updated_batch)

orderId,custId,employeeId,orderDate,requiredDate,shippedDate,shipperid,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
20248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16T00:00:00.000Z,3,99.99,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France


In [0]:
#comparing it with the silver table 
display(
    silver_salesorder_updated
    .filter(F.col("order_id") == 20248)
)

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
20248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,32.38,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France


in a real world scenario we would have received a new incremental data however when we matched it with the silver table we found that this order id already exist sso we will check the values in the new data if it ma6ches thats a duplicate record we can choose to ignore it if it doesnt matches we have to update the new record **in our case we need to update it**

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "workspace.northwind.silver_salesorder"
)

(
    silver_table.alias("target")
    .merge(
        updated_batch.alias("source"),
        "target.order_id = source.orderId"
    )
    .whenMatchedUpdate(
        set={
            "freight": "source.freight"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("workspace.northwind.silver_salesorder")
    .filter(F.col("order_id") == 20248)
)

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
20248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,99.99,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France


In [0]:
#chceking the updated value
silver_final = spark.table("workspace.northwind.silver_salesorder")

print("Silver row count:", silver_final.count())

display(
    silver_final
    .filter(
        F.col("order_id").isin(
            [20248, 20249, 20250, 20251, 20252]
        )
    )
    .orderBy("order_id")
)

Silver row count: 835


order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
20248,85,5,2006-07-04T00:00:00.000Z,2006-08-01T00:00:00.000Z,2006-07-16 00:00:00,3,99.99,Ship to 85-B,6789 rue de l'Abbaye,Reims,null,10345,France
20249,79,6,2006-07-05T00:00:00.000Z,2006-08-16T00:00:00.000Z,2006-07-10 00:00:00,1,11.61,Ship to 79-C,Luisenstr. 9012,Münster,null,10328,Germany
20250,34,4,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-12 00:00:00,2,65.83,Destination SCQXA,"Rua do Paço, 7890",Rio de Janeiro,RJ,10195,Brazil
20251,84,3,2006-07-08T00:00:00.000Z,2006-08-05T00:00:00.000Z,2006-07-15 00:00:00,1,41.34,Ship to 84-A,"3456, rue du Commerce",Lyon,null,10342,France
20252,76,4,2006-07-09T00:00:00.000Z,2006-08-06T00:00:00.000Z,2006-07-11 00:00:00,2,51.3,Ship to 76-B,"Boulevard Tirou, 9012",Charleroi,null,10318,Belgium


### checking whether the 5 new orders are in orderdetail table or not:-
this is important because our fact table consist of salesorder and orderdetail

In [0]:
new_order_ids = [20248, 20249, 20250, 20251, 20252]

orderdetail = spark.table(
    "workspace.northwind.silver_orderdetail"
)

new_order_details = orderdetail.filter(
    F.col("order_id").isin(new_order_ids)
)

print("Order-detail rows for new orders:", new_order_details.count())

display(
    new_order_details.orderBy("order_id")
)

Order-detail rows for new orders: 0


order_detail_id,order_id,product_id,unit_price,quantity,discount



### Incremental Data-Quality Observation — Orphan Orders

The 5 newly loaded orders (`20248–20252`) were successfully inserted into the Silver `salesorder` table.

However, no corresponding records were found in the Silver `orderdetail` table for these orders.

**Result:** These orders cannot currently contribute to the sales fact table because there are no order-line records associated with them.

**Action:** Do not create fact rows for these orders. Flag them for data-quality investigation.

### incremental data-quality checks


In [0]:
# checkig null
from pyspark.sql import functions as F

new_orders = silver_final.filter(
    F.col("order_id").isin([20248, 20249, 20250, 20251, 20252])
)

null_check = new_orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in new_orders.columns
])

display(null_check)

order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
0,0,0,0,0,0,0,0,0,0,0,4,0,0


In [0]:
display(
    new_orders.select(
        "order_id",
        "ship_region"
    )
)

order_id,ship_region
20249,null
20250,RJ
20251,null
20252,null
20248,null


### Null-Value Check

The 5 incremental orders contained 4 NULL values in `ship_region`.

These NULLs are considered valid because `ship_region` is an optional attribute and NULL values already exist in the source dataset.

**Result:** No data-quality failure.

### CHecking duplicates

In [0]:
print("Total records:", new_orders.count())

print(
    "Distinct order IDs:",
    new_orders.select("order_id").distinct().count()
)

Total records: 5
Distinct order IDs: 5


In [0]:
#checking for invalid values
invalid_freight = new_orders.filter(
    F.col("freight") < 0
)

print("Invalid freight records:", invalid_freight.count())

display(invalid_freight)

Invalid freight records: 0


order_id,customer_id,employee_id,order_date,required_date,shipped_date,shipper_id,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country


### Building an Automated Incremental Data Pipeline :-


In [0]:
# Data Quality Validation Function
# This function separates valid and invalid records.
# Invalid records are given a reason for why they failed the validation.

def validate_salesorder(df):

    # Rule 1: order_id cannot be NULL
    invalid_order_id = F.col("order_id").isNull()

    # Rule 2: freight cannot be negative
    invalid_freight = F.col("freight") < 0

    # Rule 3: order_date cannot be after required_date
    invalid_dates = F.col("order_date") > F.col("required_date")

    # Create an error reason for invalid records
    error_reason = (
        F.when(invalid_order_id, "Missing order_id")
         .when(invalid_freight, "Negative freight")
         .when(invalid_dates, "order_date is after required_date")
    )

    # Valid records
    valid_df = df.filter(
        ~(
            invalid_order_id |
            invalid_freight |
            invalid_dates
        )
    )

    # Invalid records with the reason for rejection
    invalid_df = (
        df.filter(
            invalid_order_id |
            invalid_freight |
            invalid_dates
        )
        .withColumn("error_reason", error_reason)
    )

    return valid_df, invalid_df

In [0]:
# Create the quarantine table for invalid salesorder records

quarantine_table = "workspace.northwind.silver_salesorder_quarantine"

if not spark.catalog.tableExists(quarantine_table):

    invalid_sample = (
        spark.table("workspace.northwind.silver_salesorder")
        .limit(0)
        .withColumn("error_reason", F.lit(None).cast("string"))
        .withColumn("quarantine_timestamp", F.current_timestamp())
    )

    invalid_sample.write.format("delta").saveAsTable(quarantine_table)

    print("Quarantine table created.")
else:
    print("Quarantine table already exists.")

Quarantine table created.


In [0]:
#The overall purpose is to Give this function an incremental CSV, and it automatically reads it, transforms it, removes duplicates within the batch, and MERGEs it into Silver. 
 
from pyspark.sql import functions as F 
from delta.tables import DeltaTable 
 
def process_salesorder_incremental(file_path): 
 
    # 1. Read incoming CSV 
    incoming_df = ( 
        spark.read 
        .option("header", True) 
        .option("inferSchema", True) 
        .csv(file_path) 
    ) 
 
    # 2. Standardize column names basically making the col name match our Silver schema 
    incoming_silver = incoming_df.select( 
        F.col("orderId").alias("order_id"), 
        F.col("custId").alias("customer_id"), 
        F.col("employeeId").alias("employee_id"), 
        F.col("orderDate").alias("order_date"), 
        F.col("requiredDate").alias("required_date"), 
        F.col("shippedDate").alias("shipped_date"), 
        F.col("shipperid").alias("shipper_id"), 
        F.col("freight"), 
        F.col("shipName").alias("ship_name"), 
        F.col("shipAddress").alias("ship_address"), 
        F.col("shipCity").alias("ship_city"), 
        F.col("shipRegion").alias("ship_region"), 
        F.col("shipPostalCode").alias("ship_postal_code"), 
        F.col("shipCountry").alias("ship_country") 
    ) 
 
    # Convert string "NULL" values into actual Spark NULL values 
    incoming_silver = incoming_silver.replace( 
        ["NULL", "null", "Null"], 
        [None, None, None] 
    ) 
 
    # 3. Remove duplicate order_ids within the incoming batch 
    incoming_silver = incoming_silver.dropDuplicates(["order_id"]) 
 
    # 4. Validate the incoming data 
    valid_df, invalid_df = validate_salesorder(incoming_silver)  #this will automatically run our validate_salesorder function 
 
    # Show how many records passed and failed the validation 
    print("Valid records:", valid_df.count()) 
    print("Invalid records:", invalid_df.count()) 
 
    # Show invalid records if any are found 
    if invalid_df.count() > 0: 
        print("Invalid records found:") 
        invalid_df.show(truncate=False)

        # Add the time when the invalid record was moved to quarantine
        invalid_df = invalid_df.withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )

        # Write invalid records into the quarantine Delta table
        invalid_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(
                "workspace.northwind.silver_salesorder_quarantine"
            )

        print("Invalid records written to quarantine table.")
 
    # 5. Get the Silver Delta table 
    silver_table = DeltaTable.forName( 
        spark, 
        "workspace.northwind.silver_salesorder"  #this tells spark Connect to our existing Silver Delta table 
    ) 
 
    # 6. MERGE incoming data into Silver 
    ( 
        silver_table.alias("target") 
        .merge( 
            valid_df.alias("source"), 
            "target.order_id = source.order_id" 
        ) 
        .whenMatchedUpdateAll() #If the order_id already exists in Silver, update the Silver record using the incoming record 
        .whenNotMatchedInsertAll() #If the order_id doesn't exist in Silver, insert the Silver record 
        .execute() 
    ) 
 
    print("Incremental load completed.")

### CReating a new Incremental QA Test File

**Duplicate records: 1 | Invalid records: 3 | Valid records: 7**

Tested: NULL conversion, deduplication, INSERT/UPDATE, negative freight, missing order_id, and invalid dates.

In [0]:
#giving the file path
process_salesorder_incremental(
    "/Volumes/workspace/northwind/raw_files/salesorder_incremental_qa_test.csv"
)

Valid records: 7
Invalid records: 3
Invalid records found:
+--------+-----------+-----------+-------------------+-------------------+------------+----------+-------+------------+-------------------------+---------+-----------+----------------+------------+---------------------------------+
|order_id|customer_id|employee_id|order_date         |required_date      |shipped_date|shipper_id|freight|ship_name   |ship_address             |ship_city|ship_region|ship_postal_code|ship_country|error_reason                     |
+--------+-----------+-----------+-------------------+-------------------+------------+----------+-------+------------+-------------------------+---------+-----------+----------------+------------+---------------------------------+
|11086.0 |49         |2          |2008-05-16 00:00:00|2008-06-13 00:00:00|NULL        |1         |-25.0  |Ship to 49-B|Via Ludovico il Moro 8901|Bergamo  |NULL       |10235           |Italy       |Negative freight                 |
|NULL    |63 